# OT Consulting – Etapa 2: Exploración de los datos
Coloca los 7 archivos (.xlsx o .csv) en la carpeta `datos/` y ejecuta las celdas en orden.

In [ ]:
import os, glob, sqlite3, warnings
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60, "display.width", 200)

def buscar(df, *palabras):
    for c in df.columns:
        if all(p.lower() in str(c).lower() for p in palabras):
            return c

def cargar(path):
    if path.lower().endswith((".xlsx", ".xls")):
        hojas = pd.read_excel(path, sheet_name=None)
        return hojas[max(hojas, key=lambda h: len(hojas[h]))]
    return pd.read_csv(path)

def tabla_de(nombre):
    n = nombre.lower()
    if "matriz" in n: return "matriz"
    for k, p in {"colab": "colaborador", "asig": "asignaci", "clientes": "cliente", "activos": "activo",
                 "deptos": "departamento", "habil": "habilidades"}.items():
        if p in n: return k

## 1. Importar los archivos

In [ ]:
dfs = {}
for a in sorted(glob.glob("datos/*.xlsx") + glob.glob("datos/*.xls") + glob.glob("datos/*.csv")):
    k = tabla_de(os.path.basename(a))
    if k and k not in dfs:
        dfs[k] = cargar(a)
for k, d in dfs.items():
    print(f"{k:10s} {d.shape[0]:>4} filas x {d.shape[1]:>3} columnas")

## 2. Perfil de cada columna

In [ ]:
def perfil(df):
    filas = []
    for c in df.columns:
        s = df[c]; t = s.dropna().astype(str)
        filas.append({"columna": c, "tipo": str(s.dtype), "nulos": int(s.isna().sum()),
                      "únicos": int(s.nunique()), "ejemplo": t.iloc[0][:30] if len(t) else None,
                      "espacios sobrantes": int((t != t.str.strip()).sum()),
                      "PENDIENTE": int(t.str.upper().str.contains("PENDIENTE").sum())})
    return pd.DataFrame(filas)

for k, d in dfs.items():
    print(f"\n=== {k.upper()} ({len(d)} filas) ===")
    display(perfil(d))

## 3. Colaboradores

In [ ]:
col = dfs["colab"].copy()
COL = {"id": buscar(col, "id"), "nombre": buscar(col, "nombre"), "estatus": buscar(col, "estatus"),
       "ingreso": buscar(col, "incorpor") or buscar(col, "ingreso"), "baja": buscar(col, "baja"),
       "depto": buscar(col, "depart"), "puesto": buscar(col, "puesto"), "ciudad": buscar(col, "ciudad")}
print(COL)
for k in ("ingreso", "baja"):
    col[COL[k]] = pd.to_datetime(col[COL[k]], errors="coerce")

est = col[COL["estatus"]].astype(str).str.strip().str.title()
activos, bajas = col[est == "Activo"], col[est == "Baja"]
print("Activos:", len(activos), "| Bajas:", len(bajas))
print("Activos sin departamento:", activos[COL["depto"]].isna().sum())
print("Activos con puesto PENDIENTE:", activos[COL["puesto"]].astype(str).str.upper().str.contains("PENDIENTE").sum())
print("Bajas sin fecha de baja:", bajas[COL["baja"]].isna().sum())
print("Activos con fecha de baja:", activos[COL["baja"]].notna().sum())
print("Baja anterior al ingreso:", (col[COL["baja"]] < col[COL["ingreso"]]).sum())

sueldos = [c for c in col.columns if "sueldo" in str(c).lower()]
for c in sueldos:
    print(f"Activos con {c} = 0 o vacío:", (pd.to_numeric(activos[c], errors="coerce").fillna(0) == 0).sum())
print("\nVariantes de ciudad:\n", col[COL["ciudad"]].astype(str).str.strip().value_counts().head(10))

In [ ]:
cols_doc = [c for c in col.columns if any(p in str(c).lower() for p in ("doc", "aviso", "formato"))]
docs = activos[cols_doc].apply(lambda s: s.astype(str).str.strip().str.upper())
completo = docs.isin(["SI", "SÍ", "NO APLICA"]).all(axis=1)
print(len(cols_doc), "columnas de documentos")
print("Activos con expediente completo:", int(completo.sum()), "de", len(activos))
print("\nDocumentos que más faltan:\n", (docs == "PENDIENTE").sum().sort_values(ascending=False).head(8))

## 4. Asignaciones

In [ ]:
asig = dfs["asig"].copy()
A = {"id": buscar(asig, "id"), "rol": buscar(asig, "rol"), "ini": buscar(asig, "inicio"), "fin": buscar(asig, "fin"),
     "cliente": next((c for c in asig.columns if "cliente" in str(c).lower()), None)}
print(A)
print("\nRoles:\n", asig[A["rol"]].astype(str).str.strip().value_counts().head(8))
print("\nFechas de inicio más repetidas:\n", asig[A["ini"]].astype(str).value_counts().head(5))
print("\nValores de fecha fin:\n", asig[A["fin"]].astype(str).value_counts().head(5))
v = asig[A["id"]].value_counts()
print("\nColaboradores con más de una asignación:\n", v[v > 1])

activos_ids = set(activos[COL["id"]].astype(str))
sin_asig = activos_ids - set(asig[A["id"]].dropna().astype(str))
print(f"\nActivos sin asignación: {len(sin_asig)} de {len(activos_ids)}")

## 5. Clientes y proyectos

In [ ]:
cli = dfs["clientes"].copy()
display(cli.head(12))
print("Nulos por columna:\n", cli.isna().sum())
c_cli = buscar(cli, "cliente")
a_cli = set(asig[A["cliente"]].dropna().astype(str).str.strip())
print("\nClientes en Asignaciones que no están en Clientes:", a_cli - set(cli[c_cli].dropna().astype(str).str.strip()))

## 6. Activos (equipo)

In [ ]:
act = dfs["activos"].copy()
AC = {"marca": buscar(act, "marca"), "costo": buscar(act, "costo"), "colab": buscar(act, "colaborador")}
print("Equipos:", len(act))
print("Costo = 0 o vacío:", (pd.to_numeric(act[AC["costo"]], errors="coerce").fillna(0) == 0).sum())
print("Equipos sin colaborador:", act[AC["colab"]].isna().sum())
print("\nMarcas:\n", act[AC["marca"]].astype(str).str.strip().value_counts())
nombres_baja = set(bajas[COL["nombre"]].astype(str).str.strip().str.upper())
n_act = act[AC["colab"]].astype(str).str.strip().str.upper()
print("\nEquipos a nombre de personas en Baja:", n_act.isin(nombres_baja).sum())

## 7. Departamentos, habilidades y matriz

In [ ]:
dep, hab, mat = dfs["deptos"], dfs["habil"], dfs["matriz"]
display(dep); print("Nulos:\n", dep.isna().sum())

c_desc = buscar(hab, "descrip")
rep = hab[hab[c_desc].duplicated(keep=False)]
print("\nHabilidades con nombre repetido:", len(rep))

id_mat = buscar(mat, "id")
cols_hab = [c for c in mat.columns if c != id_mat]
niv = pd.Series(mat[cols_hab].values.ravel()).dropna().astype(str).str.strip()
print("\nNiveles:\n", niv.value_counts())
print("Activos sin matriz de habilidades:", len(activos_ids - set(mat[id_mat].dropna().astype(str))), "de", len(activos_ids))
expertos = mat[cols_hab].apply(lambda s: s.astype(str).str.strip().str.title() == "Experto").sum()
print("Habilidades con menos de 2 expertos:", int((expertos < 2).sum()), "de", len(cols_hab))

## 8. Relaciones con SQL

In [ ]:
con = sqlite3.connect(":memory:")
col.to_sql("colaboradores", con, index=False)
asig.to_sql("asignaciones", con, index=False)
i, a = COL["id"], A["id"]

print(pd.read_sql(f'''
SELECT COUNT(*) AS asignaciones_sin_colaborador
FROM asignaciones a LEFT JOIN colaboradores c ON CAST(a."{a}" AS TEXT) = CAST(c."{i}" AS TEXT)
WHERE c."{i}" IS NULL''', con))

print(pd.read_sql(f'''
SELECT c."{COL['estatus']}" AS estatus, COUNT(*) AS personas,
       SUM(CASE WHEN x."{a}" IS NULL THEN 1 ELSE 0 END) AS sin_asignacion
FROM colaboradores c
LEFT JOIN (SELECT DISTINCT "{a}" FROM asignaciones) x ON CAST(x."{a}" AS TEXT) = CAST(c."{i}" AS TEXT)
GROUP BY 1''', con))

## 9. Primeros indicadores

In [ ]:
col["anio_ingreso"] = col[COL["ingreso"]].dt.year
r = col.groupby("anio_ingreso").apply(lambda g: pd.Series({
    "contratados": len(g),
    "bajas": (g[COL["estatus"]].astype(str).str.strip().str.title() == "Baja").sum()}))
r["% ya salió"] = (100 * r["bajas"] / r["contratados"]).round(1)
display(r)

perm = (bajas[COL["baja"]] - bajas[COL["ingreso"]]).dt.days.dropna()
print("Permanencia promedio de las bajas (días):", round(perm.mean(), 1))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
r["% ya salió"].plot(kind="bar", ax=ax[0], title="% de contratados que ya salió, por año de ingreso")
perm.plot(kind="hist", bins=12, ax=ax[1], title="Días de permanencia antes de la baja")
plt.tight_layout(); plt.show()

sti = buscar(col, "integrado") or buscar(col, "costo", "total")
costo = activos[[COL["nombre"], sti]].copy()
costo[sti] = pd.to_numeric(costo[sti], errors="coerce")
costo["costo_por_hora"] = (costo[sti] / 160).round(2)
display(costo.sort_values("costo_por_hora", ascending=False).head(10))
print("Costo mensual de la plantilla activa:", costo[sti].sum())